In [12]:
# df_enhanced_fixed.csv 


import pandas as pd
import numpy as np

# ----------------------------
# CONFIG
# ----------------------------
INPUT_PATH = "final_cleaned_dataset_2.csv"
OUTPUT_PATH = "df_enhanced_fixed.csv"

# Month name to number (case-insensitive)
month_map = {
    'january': 1, 'february': 2, 'march': 3, 'april': 4, 'may': 5, 'june': 6,
    'july': 7, 'august': 8, 'september': 9, 'october': 10, 'november': 11, 'december': 12
}

# ----------------------------
# LOAD
# ----------------------------
df = pd.read_csv(INPUT_PATH)

required_cols = [
    "Company_Cleaned","Sub_Sector","Headquarters","Amount_Cr","Funding_Round_Type",
    "Lead_Investors","Year","Month","Sector","city","state","country"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ----------------------------
# BASIC CLEANING
# ----------------------------
df["Amount_Cr"] = pd.to_numeric(df["Amount_Cr"], errors="coerce").fillna(0.0)

def clean_str_series(s, title=False):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    if title:
        s = s.str.title()
    return s

df["Company_Cleaned"] = clean_str_series(df["Company_Cleaned"], title=False)
df["Sub_Sector"]      = clean_str_series(df["Sub_Sector"], title=True)
df["Headquarters"]    = clean_str_series(df["Headquarters"], title=True)
df["Funding_Round_Type"] = clean_str_series(df["Funding_Round_Type"], title=True)
df["Lead_Investors"]  = clean_str_series(df["Lead_Investors"], title=True)
df["Sector"]          = clean_str_series(df["Sector"], title=True)
df["city"]            = clean_str_series(df["city"], title=True)
df["state"]           = clean_str_series(df["state"], title=True)
df["country"]         = clean_str_series(df["country"], title=True)

df["Lead_Investors"] = (
    df["Lead_Investors"].str.split(",")
    .apply(lambda xs: ", ".join(sorted(set([x.strip() for x in xs if x.strip()]))))
    .fillna("")
)

# ----------------------------
# DATE & TIME FIELDS
# ----------------------------
df["Month_Num"] = df["Month"].astype(str).str.lower().map(month_map).fillna(0).astype(int)

def stable_day(row):
    key = f"{row['Company_Cleaned']}|{row['Year']}|{row['Month_Num']}"
    h = abs(hash(key))
    return (h % 28) + 1

df["Day"] = df.apply(stable_day, axis=1)

df.loc[df["Month_Num"] == 0, "Month_Num"] = 1
df["Date"] = pd.to_datetime(
    dict(year=df["Year"].astype(int),
         month=df["Month_Num"].astype(int),
         day=df["Day"].astype(int)),
    errors="coerce"
)

df["FY"] = np.where(df["Month_Num"] >= 4, df["Year"], df["Year"] - 1).astype(int)

def fiscal_quarter(m):
    if m in [4,5,6]: return 1
    if m in [7,8,9]: return 2
    if m in [10,11,12]: return 3
    return 4

df["Quarter"] = df["Month_Num"].apply(fiscal_quarter).astype(int)

# ----------------------------
# COMPANY-LEVEL LAG FEATURE
# ----------------------------
df = df.sort_values(["Company_Cleaned", "Date"], kind="mergesort")
df["Cumulative_Funding_Prior"] = (
    df.groupby("Company_Cleaned")["Amount_Cr"].cumsum().shift(1).fillna(0.0)
)

# ----------------------------
# LOCAL MARKET ROLLING SIGNALS (HQ + Sector)
# ----------------------------
df["HQ_Sector"] = (
    df["Headquarters"].fillna("") + " | " + df["Sector"].fillna("")
).str.strip()

df["_MonthStart"] = df["Date"].dt.to_period("M").dt.to_timestamp()

monthly = (
    df.groupby(["HQ_Sector", "_MonthStart"])
      .agg(
          Monthly_Funding=("Amount_Cr", "sum"),
          Monthly_Rounds=("Company_Cleaned", "count"),
          Monthly_Median_Size=("Amount_Cr", "median"),
      )
      .reset_index()
)

def complete_months(g):
    idx = pd.date_range(g["_MonthStart"].min(), g["_MonthStart"].max(), freq="MS")
    g = g.set_index("_MonthStart").reindex(idx)
    g.index.name = "_MonthStart"
    g["Monthly_Funding"] = g["Monthly_Funding"].fillna(0.0)
    g["Monthly_Rounds"]  = g["Monthly_Rounds"].fillna(0)
    return g.reset_index()

monthly = monthly.groupby("HQ_Sector", group_keys=False).apply(complete_months)
monthly = monthly.sort_values(["HQ_Sector", "_MonthStart"])

def add_rollings(g):
    g = g.copy()
    g["Rolling_6m_Funding_Lagged"] = g["Monthly_Funding"].rolling(6, min_periods=1).sum().shift(1)
    g["Rolling_6m_Rounds_Lagged"]  = g["Monthly_Rounds"].rolling(6, min_periods=1).sum().shift(1)
    g["Rolling_6m_Median_Size_Lagged"] = g["Monthly_Median_Size"].rolling(6, min_periods=1).median().shift(1)

    g["Rolling_12m_Funding_Lagged"] = g["Monthly_Funding"].rolling(12, min_periods=1).sum().shift(1)
    g["Rolling_12m_Rounds_Lagged"]  = g["Monthly_Rounds"].rolling(12, min_periods=1).sum().shift(1)
    g["Rolling_12m_Median_Size_Lagged"] = g["Monthly_Median_Size"].rolling(12, min_periods=1).median().shift(1)
    return g

monthly = monthly.groupby("HQ_Sector", group_keys=False).apply(add_rollings)

df = df.merge(
    monthly[[
        "HQ_Sector","_MonthStart",
        "Rolling_6m_Funding_Lagged","Rolling_6m_Rounds_Lagged","Rolling_6m_Median_Size_Lagged",
        "Rolling_12m_Funding_Lagged","Rolling_12m_Rounds_Lagged","Rolling_12m_Median_Size_Lagged"
    ]],
    how="left",
    on=["HQ_Sector","_MonthStart"]
)


# ----------------------------------------------------------
# ✅ ADD MISSING FLAGS + FILL NA = 0
# ----------------------------------------------------------
rolling_cols = [
    "Rolling_6m_Funding_Lagged","Rolling_6m_Rounds_Lagged","Rolling_6m_Median_Size_Lagged",
    "Rolling_12m_Funding_Lagged","Rolling_12m_Rounds_Lagged","Rolling_12m_Median_Size_Lagged",
]

for col in rolling_cols:
    df[col + "_Missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(0)


# ----------------------------
# FINAL COLUMN ORDER + ROUNDING
# ----------------------------
final_cols = [
    "Company_Cleaned","Sub_Sector","Headquarters","Amount_Cr","Funding_Round_Type",
    "Lead_Investors","Year","Month","Sector","city","state","country",
    "Month_Num","Day","Date","Quarter","FY","Cumulative_Funding_Prior","HQ_Sector",
] + rolling_cols + [c + "_Missing" for c in rolling_cols]

df_final = df[final_cols].copy()

# Round float columns
float_cols = df_final.select_dtypes(include="float").columns
df_final[float_cols] = df_final[float_cols].round(2)

# ----------------------------
# SAVE
# ----------------------------
df_final.to_csv(OUTPUT_PATH, index=False)

print("✅ Saved:", OUTPUT_PATH)
print("Shape:", df_final.shape)
print("Columns:", list(df_final.columns))


C:\Users\KEYUR\AppData\Local\Temp\ipykernel_12952\4286032546.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly = monthly.groupby("HQ_Sector", group_keys=False).apply(complete_months)


✅ Saved: df_enhanced_fixed.csv
Shape: (5862, 31)
Columns: ['Company_Cleaned', 'Sub_Sector', 'Headquarters', 'Amount_Cr', 'Funding_Round_Type', 'Lead_Investors', 'Year', 'Month', 'Sector', 'city', 'state', 'country', 'Month_Num', 'Day', 'Date', 'Quarter', 'FY', 'Cumulative_Funding_Prior', 'HQ_Sector', 'Rolling_6m_Funding_Lagged', 'Rolling_6m_Rounds_Lagged', 'Rolling_6m_Median_Size_Lagged', 'Rolling_12m_Funding_Lagged', 'Rolling_12m_Rounds_Lagged', 'Rolling_12m_Median_Size_Lagged', 'Rolling_6m_Funding_Lagged_Missing', 'Rolling_6m_Rounds_Lagged_Missing', 'Rolling_6m_Median_Size_Lagged_Missing', 'Rolling_12m_Funding_Lagged_Missing', 'Rolling_12m_Rounds_Lagged_Missing', 'Rolling_12m_Median_Size_Lagged_Missing']


C:\Users\KEYUR\AppData\Local\Temp\ipykernel_12952\4286032546.py:136: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly = monthly.groupby("HQ_Sector", group_keys=False).apply(add_rollings)


In [7]:
# company_enhanced_fixed.csv 


import pandas as pd
import numpy as np

# ----------------------------
# CONFIG
# ----------------------------
INPUT = "df_enhanced_fixed.csv"
OUTPUT = "company_enhanced_fixed.csv"

# ----------------------------
# LOAD
# ----------------------------
df = pd.read_csv(INPUT, parse_dates=["Date"], dayfirst=False)

required_cols = [
    "Company_Cleaned","Sub_Sector","Headquarters","city","state","country",
    "Amount_Cr","Funding_Round_Type","Lead_Investors","Date","Quarter","FY",
    "Cumulative_Funding_Prior"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns from df_enhanced_fixed: {missing}")

# ----------------------------
# CLEANING HELPERS
# ----------------------------
def clean_str_series(s, title=True):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    if title:
        s = s.str.title()
    return s

def unique_investor_list(series):
    """Build a unique, sorted investor list from a series of comma-separated strings."""
    uniq = set()
    for val in series.fillna(""):
        for part in str(val).split(","):
            p = part.strip()
            if p:
                uniq.add(p.title())
    return ", ".join(sorted(uniq))

# Standardize text
df["Sub_Sector"]     = clean_str_series(df["Sub_Sector"])
df["Headquarters"]   = clean_str_series(df["Headquarters"])
df["city"]           = clean_str_series(df["city"])
df["state"]          = clean_str_series(df["state"])
df["country"]        = clean_str_series(df["country"])
df["Funding_Round_Type"] = clean_str_series(df["Funding_Round_Type"])
df["Lead_Investors"] = df["Lead_Investors"].fillna("").astype(str)

# Ensure numeric
df["Amount_Cr"] = pd.to_numeric(df["Amount_Cr"], errors="coerce").fillna(0.0)
df["Cumulative_Funding_Prior"] = pd.to_numeric(df["Cumulative_Funding_Prior"], errors="coerce").fillna(0.0)

# Sort for first/last calculations
df = df.sort_values(["Company_Cleaned", "Date"], kind="mergesort")

# Groupby
g = df.groupby("Company_Cleaned", sort=False)

# Precompute date aggregates safely (avoid .dt on groupby directly)
first_date = g["Date"].min()
last_date  = g["Date"].max()

# Build the frame piece by piece to avoid GroupBy.dt
company_df = pd.DataFrame({
    "Company_Cleaned": first_date.index
}).reset_index(drop=True)

# Aggregations
company_df["Total_Funding"] = g["Amount_Cr"].sum().values
company_df["Max_Funding"]   = g["Amount_Cr"].max().values
company_df["Num_Rounds"]    = g["Amount_Cr"].count().values

company_df["Sub_Sector"]   = g["Sub_Sector"].first().values
company_df["Headquarters"] = g["Headquarters"].first().values
company_df["City"]         = g["city"].first().values
company_df["State"]        = g["state"].first().values
company_df["Country"]      = g["country"].first().values

# Top 3 round types
company_df["Top_Round_Types"] = g["Funding_Round_Type"].apply(
    lambda x: ", ".join(x.value_counts().head(3).index.astype(str))
).values

# Years (unique, sorted)
company_df["Years"] = g["Date"].apply(
    lambda s: ", ".join(map(str, sorted(pd.to_datetime(s).dt.year.dropna().unique())))
).values

# Investors (unique, sorted)
company_df["Investors"] = g["Lead_Investors"].apply(unique_investor_list).values

# Last cumulative funding prior to last round (take last value)
company_df["Last_Cumulative_Prior"] = g["Cumulative_Funding_Prior"].last().values

# Dates
company_df["First_Date"] = first_date.values
company_df["Last_Date"]  = last_date.values

# Quarters/FYs (unique sets)
company_df["Quarters"] = g["Quarter"].apply(
    lambda s: ", ".join(map(str, sorted(pd.Series(s).dropna().unique())))
).values

company_df["FYs"] = g["FY"].apply(
    lambda s: ", ".join(map(str, sorted(pd.Series(s).dropna().unique())))
).values

# First_Year, Last_Year from dates
company_df["First_Year"] = pd.to_datetime(company_df["First_Date"]).dt.year
company_df["Last_Year"]  = pd.to_datetime(company_df["Last_Date"]).dt.year

# ----------------------------
# ROUND NUMERIC COLUMNS
# ----------------------------
float_cols = company_df.select_dtypes(include="float").columns
company_df[float_cols] = company_df[float_cols].round(2)

# ----------------------------
# SORT
# ----------------------------
company_df = company_df.sort_values("Total_Funding", ascending=False)

# ----------------------------
# SAVE
# ----------------------------
company_df.to_csv(OUTPUT, index=False)

print("✅ Saved:", OUTPUT)
print("Shape:", company_df.shape)
print("Columns:", list(company_df.columns))


✅ Saved: company_enhanced_fixed.csv
Shape: (4183, 19)
Columns: ['Company_Cleaned', 'Total_Funding', 'Max_Funding', 'Num_Rounds', 'Sub_Sector', 'Headquarters', 'City', 'State', 'Country', 'Top_Round_Types', 'Years', 'Investors', 'Last_Cumulative_Prior', 'First_Date', 'Last_Date', 'Quarters', 'FYs', 'First_Year', 'Last_Year']


In [8]:
# investor_enhanced_fixed.csv 

import pandas as pd
import numpy as np

# ----------------------------
# CONFIG
# ----------------------------
INPUT = "df_enhanced_fixed.csv"
OUTPUT = "investor_enhanced_fixed.csv"

# ----------------------------
# LOAD
# ----------------------------
df = pd.read_csv(INPUT, parse_dates=["Date"], dayfirst=False)

required_cols = [
    "Company_Cleaned","Sub_Sector","city","state","country",
    "Amount_Cr","Funding_Round_Type","Lead_Investors","Date"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns from df_enhanced_fixed: {missing}")

# ----------------------------
# CLEANING HELPERS
# ----------------------------
def clean_str_series(s, title=True):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    if title:
        s = s.str.title()
    return s

df["Sub_Sector"]        = clean_str_series(df["Sub_Sector"])
df["Funding_Round_Type"] = clean_str_series(df["Funding_Round_Type"])
df["Lead_Investors"]    = df["Lead_Investors"].fillna("").astype(str)

df["Amount_Cr"] = pd.to_numeric(df["Amount_Cr"], errors="coerce").fillna(0.0)

# ----------------------------
# EXPAND INVESTORS
# ----------------------------
df["Investor_List"] = df["Lead_Investors"].apply(
    lambda x: sorted(set([p.strip().title() for p in str(x).split(",") if p.strip()]))
)

inv_df = df.explode("Investor_List")
inv_df = inv_df.dropna(subset=["Investor_List"])

# ----------------------------
# GROUPBY INVESTOR
# ----------------------------
g = inv_df.groupby("Investor_List", sort=False)

# Basic aggregates
investor_df = pd.DataFrame({
    "Investor": g.size().index
}).reset_index(drop=True)

investor_df["Total_Invested"]  = g["Amount_Cr"].sum().values
investor_df["Num_Investments"] = g["Company_Cleaned"].nunique().values

# Last investment
last_date = g["Date"].max()
investor_df["Last_Date"] = last_date.values

# Top sub-sectors
investor_df["Top_Sub_Sectors"] = g["Sub_Sector"].apply(
    lambda x: ", ".join(x.value_counts().head(3).index.astype(str))
).values

# Top stages (funding round type)
investor_df["Top_Stages"] = g["Funding_Round_Type"].apply(
    lambda x: ", ".join(x.value_counts().head(3).index.astype(str))
).values

# Top cities
investor_df["Top_Cities"] = g["city"].apply(
    lambda x: ", ".join(x.value_counts().head(3).index.astype(str))
).values

# ----------------------------
# Avg HHI & Avg Unique lead count
# ----------------------------
# HHI per company for the investor
# HHI = sum(share^2) across co-leads per round
# share = 1/(num_leads)
# so HHI = sum(1/n^2) with n leads

def compute_hhi(list_str):
    """Given comma-separated investors → compute HHI"""
    parts = [p.strip() for p in str(list_str).split(",") if p.strip()]
    n = len(parts)
    if n == 0:
        return 0
    return sum([(1/n)**2]*n)   # = n * (1/n^2) = 1/n

inv_df["HHI"] = inv_df["Lead_Investors"].apply(compute_hhi)
inv_df["Unique_Leads_Count"] = inv_df["Lead_Investors"].apply(
    lambda x: len([p.strip() for p in str(x).split(",") if p.strip()])
)

investor_df["Avg_HHI"] = g["HHI"].mean().values
investor_df["Avg_Unique_Leads"] = g["Unique_Leads_Count"].mean().values

# ----------------------------
# LAST_INVESTMENT (Full date)
# ----------------------------
investor_df["Last_Investment"] = pd.to_datetime(investor_df["Last_Date"])

# ----------------------------
# ROUND NUMERIC
# ----------------------------
float_cols = investor_df.select_dtypes(include="float").columns
investor_df[float_cols] = investor_df[float_cols].round(2)

# ----------------------------
# SORT
# ----------------------------
investor_df = investor_df.sort_values("Total_Invested", ascending=False)

# ----------------------------
# SAVE
# ----------------------------
investor_df.to_csv(OUTPUT, index=False)

print("✅ Saved:", OUTPUT)
print("Shape:", investor_df.shape)
print("Columns:", list(investor_df.columns))


✅ Saved: investor_enhanced_fixed.csv
Shape: (5302, 10)
Columns: ['Investor', 'Total_Invested', 'Num_Investments', 'Last_Date', 'Top_Sub_Sectors', 'Top_Stages', 'Top_Cities', 'Avg_HHI', 'Avg_Unique_Leads', 'Last_Investment']


In [11]:
# market_signals_summary_fixed.csv 

import pandas as pd
import numpy as np

# ----------------------------
# CONFIG
# ----------------------------
INPUT = "df_enhanced_fixed.csv"
OUTPUT = "market_signals_summary_fixed.csv"

# ----------------------------
# LOAD
# ----------------------------
df = pd.read_csv(INPUT, parse_dates=["Date"], dayfirst=False)

required_cols = ["Headquarters", "Sector", "Amount_Cr", "Date"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns from df_enhanced_fixed: {missing}")

# Clean
df["Headquarters"] = df["Headquarters"].astype(str).str.title().str.strip()
df["Sector"]       = df["Sector"].astype(str).str.title().str.strip()
df["Amount_Cr"]    = pd.to_numeric(df["Amount_Cr"], errors="coerce").fillna(0.0)

# Month start
df["_MonthStart"] = df["Date"].dt.to_period("M").dt.to_timestamp()  # first of month

# HQ+Sector grouping key
df["HQ_Sector"] = (df["Headquarters"].fillna("") + " | " + df["Sector"].fillna("")).str.strip()

# ----------------------------
# Monthly aggregation
# ----------------------------
monthly = (
    df.groupby(["HQ_Sector", "_MonthStart"])
      .agg(
          Monthly_Funding=("Amount_Cr", "sum"),
          Monthly_Rounds=("Headquarters", "count"),
          Monthly_Median_Size=("Amount_Cr", "median"),
      )
      .reset_index()
)

# Ensure complete monthly range per HQ-Sector
def complete_months(g):
    idx = pd.date_range(g["_MonthStart"].min(), g["_MonthStart"].max(), freq="MS")
    g = g.set_index("_MonthStart").reindex(idx)
    g.index.name = "_MonthStart"
    g["Monthly_Funding"] = g["Monthly_Funding"].fillna(0.0)
    g["Monthly_Rounds"]  = g["Monthly_Rounds"].fillna(0)
    return g.reset_index()

monthly = (
    monthly.groupby("HQ_Sector", group_keys=False)
           .apply(complete_months)
)

monthly = monthly.sort_values(["HQ_Sector", "_MonthStart"])

# ----------------------------
# Rolling (lagged) calculations
# ----------------------------
def add_rollings(g):
    g = g.copy()
    g["Rolling_6m_Funding"]      = g["Monthly_Funding"].rolling(6, min_periods=1).sum().shift(1)
    g["Rolling_6m_Rounds"]       = g["Monthly_Rounds"].rolling(6, min_periods=1).sum().shift(1)
    g["Rolling_6m_Median_Size"]  = g["Monthly_Median_Size"].rolling(6, min_periods=1).median().shift(1)

    g["Rolling_12m_Funding"]     = g["Monthly_Funding"].rolling(12, min_periods=1).sum().shift(1)
    g["Rolling_12m_Rounds"]      = g["Monthly_Rounds"].rolling(12, min_periods=1).sum().shift(1)
    g["Rolling_12m_Median_Size"] = g["Monthly_Median_Size"].rolling(12, min_periods=1).median().shift(1)
    return g

monthly = monthly.groupby("HQ_Sector", group_keys=False).apply(add_rollings)

# ----------------------------
# Final high-level summary
# ----------------------------
summary = (
    monthly.groupby("HQ_Sector")
           .agg(
               Avg_6m_Funding      = ("Rolling_6m_Funding", "mean"),
               Avg_6m_Rounds       = ("Rolling_6m_Rounds", "mean"),
               Avg_6m_Median_Size  = ("Rolling_6m_Median_Size", "mean"),
               Avg_12m_Funding     = ("Rolling_12m_Funding", "mean"),
               Avg_12m_Rounds      = ("Rolling_12m_Rounds", "mean"),
               Avg_12m_Median_Size = ("Rolling_12m_Median_Size", "mean"),
               Total_Funding       = ("Monthly_Funding", "sum"),
               Total_Rounds        = ("Monthly_Rounds", "sum"),
           )
           .reset_index()
)

# Split HQ_Sector → Headquarters + Sector columns
summary[["Headquarters", "Sector"]] = summary["HQ_Sector"].str.split("\|", expand=True)
summary["Headquarters"] = summary["Headquarters"].str.strip()
summary["Sector"]       = summary["Sector"].str.strip()

# Reorder cols
cols = [
    "Headquarters", "Sector",
    "Avg_6m_Funding", "Avg_6m_Rounds", "Avg_6m_Median_Size",
    "Avg_12m_Funding", "Avg_12m_Rounds", "Avg_12m_Median_Size",
    "Total_Funding", "Total_Rounds",
]
summary = summary[cols]

# ---------------------------------------------------
# ✅ MISSING VALUE HANDLING
# ---------------------------------------------------
rolling_cols = [
    "Avg_6m_Funding", "Avg_6m_Rounds", "Avg_6m_Median_Size",
    "Avg_12m_Funding", "Avg_12m_Rounds", "Avg_12m_Median_Size",
]

for col in rolling_cols:
    # Missing flag
    summary[col + "_Missing"] = summary[col].isna().astype(int)
    # Fill NaN with 0
    summary[col] = summary[col].fillna(0)

# Round numeric
float_cols = summary.select_dtypes(include="float").columns
summary[float_cols] = summary[float_cols].round(2)

# ----------------------------
# SAVE
# ----------------------------
summary.to_csv(OUTPUT, index=False)

print("✅ Saved:", OUTPUT)
print("Shape:", summary.shape)
print("Columns:", list(summary.columns))


C:\Users\KEYUR\AppData\Local\Temp\ipykernel_12952\2873299555.py:55: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(complete_months)


✅ Saved: market_signals_summary_fixed.csv
Shape: (632, 16)
Columns: ['Headquarters', 'Sector', 'Avg_6m_Funding', 'Avg_6m_Rounds', 'Avg_6m_Median_Size', 'Avg_12m_Funding', 'Avg_12m_Rounds', 'Avg_12m_Median_Size', 'Total_Funding', 'Total_Rounds', 'Avg_6m_Funding_Missing', 'Avg_6m_Rounds_Missing', 'Avg_6m_Median_Size_Missing', 'Avg_12m_Funding_Missing', 'Avg_12m_Rounds_Missing', 'Avg_12m_Median_Size_Missing']


C:\Users\KEYUR\AppData\Local\Temp\ipykernel_12952\2873299555.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly = monthly.groupby("HQ_Sector", group_keys=False).apply(add_rollings)
